In [17]:
# load json
import json
with open("narrtaive_structure.json", "r") as file:
    structure_instructions_list = json.load(file)

In [6]:
import pygraphviz as pgv
import json
import random
from IPython.display import display, Markdown
import json
from gpt_cartographer import make_graph
# ignore warnings
import warnings
warnings.filterwarnings("ignore")

def save_graph_json(graph_json, output_file="graph.json"):
    with open(output_file, "w") as f:
        f.write(graph_json)
        print("Graph json saved to", output_file)


def plot_graph(graph_json, output_file="output.png", direction="LR"):

    # Parse the JSON if it's a string
    if isinstance(graph_json, str):
        graph_data = json.loads(graph_json)
    else:
        graph_data = graph_json

    # Create a new graph
    G = pgv.AGraph(directed=True, rankdir=direction)

    # Add nodes
    for node in graph_data["nodes"]:
        label = ", ".join(node["time_period"]) + " "+ node["label"] 
        G.add_node(node["index"], label=label)

    # Add edges
    for edge in graph_data["edges"]:
        G.add_edge(edge["start_node"], edge["end_node"], label=edge["labels"])

    # Draw the graph
    G.draw(output_file, prog="dot")
    print("Graph saved to", output_file)


def retrieve_section_data(section_name, data, seed=None):
    """
    Retrieve specified section data with random selections.
    
    Parameters:
        section_name (str): The name of the section to retrieve data from.
        data (dict): The data dictionary containing section data.
        seed (int, optional): The seed for the random number generator.
        
    Returns:
        dict: A dictionary containing randomly selected entities and themes.
    """
    # Set the seed
    random.seed(seed)

    section_data = data.get(section_name, {})
    entities_data = section_data.get("Entities", {})
    themes_data = section_data.get("Themes", {})

    # Randomly select entity types
    selected_entity_types = random.sample(list(entities_data.keys()), min(len(entities_data), random.randint(2, len(entities_data))))

    new_entities_data = {}
    for entity_type in selected_entity_types:
        entities = entities_data[entity_type]
        if isinstance(entities, list):
            new_entities_data[entity_type] = random.sample(entities, min(len(entities), random.randint(2, len(entities))))

    # Randomly select themes and convert them to a list of dictionaries
    if isinstance(themes_data, list):
        selected_themes = random.sample(themes_data, random.randint(2, len(themes_data)))
        themes_dict_list = [{"name": theme["name"], "description": theme["description"]} for theme in selected_themes]
    else:
        themes_dict_list = []

    print(f"Selected {len(new_entities_data)} entity types and {len(themes_dict_list)} themes from {section_name} section.")
    print(f"Selected entity types: {new_entities_data.keys()}")
    print(f"Selected entities names: {[entity['name'] for entity_type in new_entities_data for entity in new_entities_data[entity_type]]}")
    print(f"Selected themes names: {[theme['name'] for theme in themes_dict_list]}")

    return {
        "Entities": new_entities_data,
        "Themes": themes_dict_list,
        "Time": section_data.get("Time", ""),
    }

def parse_context_data(context_data):
    """Parse the context data into string format for entities, themes, and time range."""
    print("Parsing context data...")
    # Flatten the list of lists for entities
    entities_list = [entity for entity_group in context_data.get("Entities", {}).values() for entity in entity_group]
    
    entities_str = "\n".join(
        [f"- {entity['name']} : {entity['description']}" for entity in entities_list])
    
    themes_str = "\n".join(
        [f"- {theme['name']} : {theme['description']}" for theme in context_data.get("Themes", [])])
    
    time_str = " to ".join(context_data.get("Time", []))


    print(f"Entities str:\n{entities_str}\n")
    print(f"Themes str:\n{themes_str}\n")
    print(f"Time str:\n{time_str}\n")

    
    return entities_str, themes_str, time_str

def parse_instruction(instruction):
    name = instruction["name"]
    description = instruction["description"]
    structure = instruction["structure"]
    instruction_text = instruction["instructions"]

    parsed_string = f"{name}\nDescription: {description}\nStructure example: {structure}\nInstructions: {instruction_text}\n"
    return parsed_string

def generate_graph(section_name, structure_name, seed=None):
    """
    Generate a graph based on entities, themes, and time data from 'narrative_context.json'.
    
    Parameters:
        section (str): Section of the narrative context to use.
        structure_instructions (str): Instructions for the graph structure.
        seed (int, optional): Seed for reproducibility.

    Returns:
        Graph, dict: Returns the generated graph and its JSON representation.
    """

    # Load data from 'narrative_context.json'
    with open("narrative_context.json", "r") as file:
        context_data = json.load(file)
    
    # Set seed for reproducibility
    if seed:
        random.seed(seed)

    # Retrieve data based on section_name
    print(f"Retrieving data from '{section_name}' section...")
    context = retrieve_section_data(section_name=section_name, data=context_data,seed=seed)

    # parse data as string for make_graph
    entities_str, themes_str, time_str = parse_context_data(context)
  

    structure_instructions = next((structure for structure in structure_instructions_list if structure["name"] == structure_name), None)

    if not structure_instructions:
        raise ValueError(f"Could not find structure instructions with name '{structure_name}'")
    
    structure_instruction_str = parse_instruction(structure_instructions)

    general_instructions = "Nodes suffixes represent the chronological order. Nodes with the same prefix should have at least one theme in common. Nodes with smaller suffixes should have an earlier time_period."

    final_structure_instructions = f"{general_instructions}\n\n{structure_instruction_str}"
    print(f"Structure Instructions:\n{final_structure_instructions}\n")


    print("Generating graph...")

    # Generate graph
    graph = make_graph(
        time_range=time_str,
        entities=entities_str,
        themes=themes_str,
        model_name="gpt-4",
        temperature=0,
        structure_instructions=final_structure_instructions,
    )

    print("Graph generated!")

    out_graphs_path = f"graphs/{section_name}_{structure_name}_{seed}"

    
    display(Markdown(f"```json\n{graph.json(indent=2)}\n```"))
    plot_graph(graph.json(), f"{out_graphs_path}.png", direction="TB")
    save_graph_json(graph.json(), f"{out_graphs_path}.json")

    return graph

    

In [7]:
graph = generate_graph(section_name="Politics", structure_name="Diamond Structure", seed=42)

Retrieving data from 'Politics' section...
Selected 4 entity types and 2 themes from Politics section.
Selected entity types: dict_keys(['People', 'Events', 'Places', 'Organizations'])
Selected entities names: ['Martina Vales', 'Chancellor Eric Weber', 'The Capital City Debate', 'The Liberty Rally', 'Capital City', 'The Grand Assembly Hall', 'Bridgeside', 'The Patriot Press', 'International Alliance Forum']
Selected themes names: ['Corruption & Transparency', 'Media Influence']
Parsing context data...
Entities str:
- Martina Vales : A veteran journalist with insider knowledge of the political arena.
- Chancellor Eric Weber : A foreign diplomat looking to foster ties with other nations.
- The Capital City Debate : A televised event where key political figures square off.
- The Liberty Rally : Organized by activists demanding political reforms.
- Capital City : The political epicenter, home to major government institutions.
- The Grand Assembly Hall : Where major political decisions are 

```json
{
  "nodes": [
    {
      "index": 1,
      "label": "The Unveiling",
      "time_period": [
        "2020-03-04",
        "2020-06-15"
      ],
      "themes": [
        "Corruption & Transparency"
      ],
      "entity": [
        "Martina Vales",
        "The Patriot Press",
        "Capital City"
      ]
    },
    {
      "index": 2,
      "label": "The Diplomat's Gambit",
      "time_period": [
        "2020-06-16",
        "2021-01-20"
      ],
      "themes": [
        "Media Influence"
      ],
      "entity": [
        "Chancellor Eric Weber",
        "The Capital City Debate",
        "International Alliance Forum"
      ]
    },
    {
      "index": 3,
      "label": "The Rally Cry",
      "time_period": [
        "2021-01-21",
        "2021-08-30"
      ],
      "themes": [
        "Corruption & Transparency"
      ],
      "entity": [
        "The Liberty Rally",
        "Bridgeside",
        "The Grand Assembly Hall"
      ]
    },
    {
      "index": 4,
      "label": "The Borderline",
      "time_period": [
        "2021-08-31",
        "2022-07-05"
      ],
      "themes": [
        "Media Influence"
      ],
      "entity": [
        "Bridgeside",
        "The Patriot Press",
        "Chancellor Eric Weber"
      ]
    },
    {
      "index": 5,
      "label": "The Final Verdict",
      "time_period": [
        "2022-07-06",
        "2022-07-05"
      ],
      "themes": [
        "Corruption & Transparency"
      ],
      "entity": [
        "Martina Vales",
        "The Grand Assembly Hall",
        "Capital City"
      ]
    }
  ],
  "edges": [
    {
      "start_node": 1,
      "end_node": 2,
      "labels": "Martina Vales uncovers a scandal, leading to Chancellor Weber's involvement in the Capital City Debate"
    },
    {
      "start_node": 1,
      "end_node": 3,
      "labels": "The scandal sparks the Liberty Rally, leading to political upheaval in Bridgeside"
    },
    {
      "start_node": 2,
      "end_node": 4,
      "labels": "Chancellor Weber's actions at the debate influence the situation in Bridgeside"
    },
    {
      "start_node": 3,
      "end_node": 5,
      "labels": "The Liberty Rally's impact leads to a final decision in the Grand Assembly Hall"
    },
    {
      "start_node": 4,
      "end_node": 5,
      "labels": "The situation in Bridgeside influences the final decision in the Grand Assembly Hall"
    }
  ]
}
```

Graph saved to graphs/Politics_Diamond Structure_42.png
Graph json saved to graphs/Politics_Diamond Structure_42.json


In [15]:
def create_graph(narrative):
    graph = {node: [] for node in narrative['nodes']}
    for edge in narrative['edges']:
        graph[edge['from']].append(edge['to'])
    return graph

def traverse_each_path(narrative):
    graph = create_graph(narrative)
    
    # Find the starting nodes (nodes without incoming edges)
    start_nodes = set(narrative['nodes']) - set(edge['to'] for edge in narrative['edges'])

    # Depth-first search to explore all paths
    def dfs(node, path):
        if not graph[node]:
            print(" -> ".join(path))
            print("="*20)
            return
        for neighbor in graph[node]:
            dfs(neighbor, path + [neighbor])

    # Traverse each path starting from the start nodes
    for start_node in start_nodes:
        dfs(start_node, [start_node])

# Example
narrative_obj =  {
        "name": "Diamond",
        "description": "Begins and concludes with Theme A. In between, it branches into two paths: one following Theme B and the other Theme C.",
        "structure": "A1 -> B2 -> A3; A1 -> C2 -> A3",
        "instructions": "Contains ONLY 5 nodes and 4 edges in total.",
        "nodes": ["A1", "B2", "A3", "C2"],
        "edges": [
            {"from": "A1", "to": "B2"},
            {"from": "B2", "to": "A3"},
            {"from": "A1", "to": "C2"},
            {"from": "C2", "to": "A3"},
        ]
    }




A1 -> B2 -> A3
A1 -> C2 -> A3


In [19]:
# for each of structure_instructions_list, draw graph about "Politics"

for each in structure_instructions_list:
    structure_name = each["name"]
    print(structure_name)
    traverse_each_path(each)
    print("-----")
    # generate_graph(section_name="Politics", structure_name=structure_name, seed=42)
    


Linear
A1 -> A2 -> A3 -> A4
-----
Divergence
A1 -> A2 -> A3
A1 -> A2 -> B3
-----
Convergence
A1 -> A2 -> A3
B1 -> A2 -> A3
-----
Diamond
A1 -> B2 -> A3
A1 -> C2 -> A3
-----
Hanger
A1 -> A2 -> A3
A1 -> B2 -> A3
-----
Parallelogram
A1 -> A2 -> B3
A1 -> B2 -> B3
-----
